In [ ]:
#%%writefile C:\Users\neele\Music\Travscape\agents\GlobalState.py

from typing import TypedDict, Annotated, List, Optional
from langgraph.graph.message import add_messages
from langchain_core.messages import SystemMessage, AIMessage, HumanMessage
from pydantic import BaseModel, Field


# State
class AgentState(TypedDict):
  messages: Annotated[list, add_messages]
  need_clarification: bool
  user_request: list

# Structured Outputs
class TripRequest(BaseModel):
    destination: Optional[str] = None
    dates: Optional[str] = None
    travelers: Optional[int] = None
    budget: Optional[str] = None
    purpose: Optional[str] = None
    preferences: Optional[str] = None

class chatbot_output(BaseModel):
  user_request:List[TripRequest]= Field(default_factory=list, description="List of all the requests that the user has. Also includes the details of the trip.")
  need_clarification: bool = Field(False, description="True if the chatbot needs clarification from the user. False otherwise.")
  clarification_question: Optional[str] = Field(None, description="The clarification question that the chatbot needs to ask the user. None, if no clarification is needed.")
  chatbot_reply: Optional[str] = Field(None, description="The chatbot's response to the user.")


Writing C:\Users\neele\Music\Travscape\agents\Agentstate.py


In [ ]:
from agents.utils import get_today_str
from langgraph.graph import StateGraph, END, START
from langgraph.types import Command
from langgraph.checkpoint.memory import InMemorySaver
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from agents.GlobalState import AgentState, chatbot_output
from agents.prompts import chatbot_message

load_dotenv(override=True)

chatbot = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
    max_tokens=None,
    timeout=None,
    max_retries=2
)

chatbot_with_output = chatbot.with_structured_output(chatbot_output)

#System message initialization with today's date
system_message = chatbot_message.format(
    date=get_today_str()
)

def Chatbot_node(state: AgentState) -> AgentState:
  
  found_system_message = False
  messages = state["messages"]
  for message in messages:
      if isinstance(message, SystemMessage):
          message.content = system_message
          found_system_message = True
  
  if not found_system_message:
      messages = [SystemMessage(content=system_message)] + messages

  try:
    response = chatbot_with_output.invoke(messages)

    # Create the main response message
    ai_response = AIMessage(content=response.chatbot_reply)
    updated_messages = messages + [ai_response]

    # Only append clarification question if it exists and has meaningful content
    if (response.need_clarification and 
        response.clarification_question and 
        response.clarification_question.strip() and
        response.clarification_question.lower() not in ["none", "null"] and
        "none, if no clarification is needed" not in response.clarification_question.lower()):
        
        # Combine the reply and clarification question in a single message
        combined_content = f"{response.chatbot_reply}\n\n{response.clarification_question}"
        ai_response = AIMessage(content=combined_content)
        updated_messages = messages + [ai_response]

    trip_requests = [req.model_dump() for req in response.user_request] if response.user_request else []

    return Command(
        update={
            "messages": updated_messages,
            "need_clarification": response.need_clarification,
            "user_request": trip_requests
        }
    )
  
  except Exception as e:
    print(f"Error: {e}")
    error_message = AIMessage(content="Sorry, I encountered an error. Please try again.")
    return Command(
        update={"messages": messages + [error_message]}
    )

checkpointer = InMemorySaver()

chatbot_builder = StateGraph(AgentState)

chatbot_builder.add_node("Chatbot", Chatbot_node)

chatbot_builder.add_edge(START, "Chatbot")
chatbot_builder.add_edge("Chatbot", END)

graph = chatbot_builder.compile(checkpointer=checkpointer)

In [ ]:
from IPython.display import Image, display
display(Image(graph.get_graph(xray=True).draw_mermaid_png()))

In [ ]:
import gradio as gr
config = {"configurable": {"thread_id": "1"}}

def chat(message, history):
    try:
        # Extract user message content
        user_message = message if isinstance(message, str) else message.get("text", "")
        
        # Invoke the graph
        result = graph.invoke(
            {"messages": [HumanMessage(content=user_message)]}, 
            config=config
        )
        
        # Return the assistant's response in proper format
        assistant_response = result["messages"][-1].content
        return assistant_response
        
    except Exception as e:
        print(f"Chat error: {e}")
        return "Sorry, I encountered an error. Please try again."

gr.ChatInterface(
    chat, 
    type="messages",
    title="Trav - Your Travel Planning Assistant",
    description="Hi! I'm Trav, ready to help you plan your next adventure!"
).launch()